# 02 - Data preprocessing and feature engineering
Turns the raw CSV into a clean dataset and a fixed **70 / 15 / 15 train / validation / test** split that every later notebook re-uses.



In [6]:
import sys, warnings
sys.path.append("..")                      # lets the notebook import the reusable code in ../src
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src import config, preprocess as P, models as M
from src.evaluate import regression_metrics, permutation_table
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format
config.PLOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Clean the raw CSV
This section cleans the raw dataset by removing records with missing or invalid prices and duplicate listings. It identifies the data source, extracts RAM and storage information, and standardizes brand, condition, model name and location values. Suspicious prices are also flagged for further checking.

In [7]:
clean, report, splits = P.run_preprocessing()
report

27 suspicious prices flagged (kept unless drop_suspicious=True).


,step,rows_after
0,raw file,5158
1,drop missing price,5158
2,drop price <= 0,5158
3,drop duplicate listings,4432
4,final clean dataset,4432


In [8]:
clean.head(10)

,id,title,brand,model,condition,storage_gb,ram_gb,location,source,price,suspicious_price
0,1,Apple iPhone 15 (Brand New),Apple,apple iphone 15,Brand New,NaN,NaN,Kandy,ikman,200000,False
1,2,Samsung Galaxy A56 5G 8GB 256GB (Brand New),Samsung,samsung galaxy a56 5g,Brand New,256.00,8.00,Colombo,ikman,95989,False
2,3,Apple iPhone 14 128GB (Used),Apple,apple iphone 14,Used,128.00,NaN,Colombo,ikman,119999,False
3,4,Infinix Hot 60i 128GB 6GBRam Blue (Brand New),Infinix,infinix hot 60i,Brand New,128.00,6.00,Colombo,ikman,39990,False
4,5,Apple iPhone 14 Pro (Used),Apple,apple iphone 14 pro,Used,NaN,NaN,Kurunegala,ikman,170000,False
5,6,Apple iPhone 11 Pro 256GB (Used),Apple,apple iphone 11 pro,Used,256.00,NaN,Gampaha,ikman,89998,False
6,7,Apple iPhone 15 Pro 256GB (Used),Apple,apple iphone 15 pro,Used,256.00,NaN,Kandy,ikman,224900,False
7,8,Apple iPhone 15 Pro 5G- 4k cinematic (Used),Apple,apple iphone 15 pro 5g,Used,NaN,NaN,Colombo,ikman,198000,False
8,9,Apple iPhone 12 mini 128GB Blue (Used),Apple,apple iphone 12 mini,Used,128.00,NaN,Colombo,ikman,69990,False
9,10,Vivo Y93 6GB/ 128GB (Used),Vivo,vivo y93,Used,128.00,6.00,Colombo,ikman,8900,False


## 2. Review suspicious prices (flagged, **not** deleted automatically)
A price is flagged when it is below 40% or above 250% of the median of the same brand + model + condition group (groups of 5+), or below Rs. 3,000. Open the ad, decide whether it is a data error (remove it) or a real but unusual price (keep it).

In [9]:
sus = clean[clean["suspicious_price"]]
print(len(sus), "flagged rows")
sus[["id", "title", "model", "condition", "price", "source"]].head(30)

27 flagged rows


,id,title,model,condition,price,source
21,24,Nokia 106 (Used),nokia 106,Used,2500,ikman
93,99,Nokia 105 (Brand New),nokia 105,Brand New,2990,ikman
358,377,Xiaomi Redmi 15 128GB/256GB (Brand New),xiaomi redmi 15,Brand New,19100,ikman
395,420,Samsung Galaxy S24 Ultra (Brand New),samsung galaxy s24 ultra,Brand New,32360,ikman
489,520,Nokia 105 (Used),nokia 105,Used,1000,ikman
823,876,Corn Button Phone (Used),other corn button,Used,2500,ikman
916,976,Nokia 105 Dubai Series (Brand New),nokia 105 dubai series,Brand New,2990,ikman
1133,1206,Samsung Galaxy S21 Ultra (Used),samsung galaxy s21 ultra,Used,30000,ikman
1142,1215,Apple iPhone 14 Pro Max (Used),apple iphone 14 pro max,Used,70000,ikman
1720,1854,Apple iPhone 17 Pro Max 256GB (Brand New),apple iphone 17 pro max,Brand New,60000,ikman


in `src/config.py

## 3. Missing values in the model features
checks missing values in the features used for machine learning. Missing RAM and storage values are kept because they may still provide useful information. The model pipeline handles these missing values using median imputation and missing-value indicators.

In [10]:
clean[config.FEATURES + [config.TARGET]].isnull().sum()

brand            0
model            0
condition        0
location         0
source           0
storage_gb    1613
ram_gb        3313
price            0
dtype: int64

## 4. Features and target
| | |
|---|---|
| Target | `price` (LKR) |
| Categorical | `brand`, `model` (cleaned, brand + model name), `condition`, `location`, `source` |
| Numerical | `storage_gb`, `ram_gb` |
| Not used | `id`, `title`, `description`, `seller_name`, `ad_url`, `price_raw` (would leak the answer), `is_member`, `is_featured` |

## 5. Split sizes
divides the cleaned dataset into three separate subsets for training, validation and testing. It also compares the average prices and ikman data proportions in each subset.

In [11]:
for k, v in splits.items():
    print(f"{k:5s} {len(v):5d} rows | mean price {v['price'].mean():>10,.0f} | ikman share {(v['source']=='ikman').mean():.2f}")

train  3102 rows | mean price    102,561 | ikman share 0.99
val     665 rows | mean price    103,052 | ikman share 0.99
test    665 rows | mean price    101,315 | ikman share 0.99


### Leakage check: the same advert must not sit in both train and test

In [12]:
key = lambda d: d["title"].astype(str) + "|" + d["price"].astype(str) + "|" + d["location"].astype(str)
tr, va, te = splits["train"], splits["val"], splits["test"]
print("identical listings in train & test:", len(set(key(tr)) & set(key(te))))
print("identical listings in train & val :", len(set(key(tr)) & set(key(va))))

identical listings in train & test: 0
identical listings in train & val : 0


## 6. What the model pipeline does to the features
demonstrates how categorical and numerical features are converted into a format suitable for machine learning models. The preprocessing steps are fitted using training data only to prevent data leakage.

In [13]:
from src.features import build_preprocessor
X_tr, y_tr = P.xy(tr)
prep = build_preprocessor().fit(X_tr)
Xt = prep.transform(X_tr)
print("encoded feature matrix:", Xt.shape)
list(prep.get_feature_names_out())[:15]

encoded feature matrix: (3102, 234)


['num__storage_gb',
 'num__ram_gb',
 'num__missingindicator_storage_gb',
 'num__missingindicator_ram_gb',
 'cat__brand_Apple',
 'cat__brand_Google',
 'cat__brand_Honor',
 'cat__brand_Huawei',
 'cat__brand_Infinix',
 'cat__brand_Lg',
 'cat__brand_Nokia',
 'cat__brand_Nothing',
 'cat__brand_Oneplus',
 'cat__brand_Oppo',
 'cat__brand_Other']

## 7. Why a log-transformed target?
compares the distribution of original mobile phone prices with the log-transformed prices. Mobile phone prices can vary significantly, with a small number of expensive phones creating a right-skewed distribution.

Applying a logarithmic transformation helps reduce this skewness and makes the target distribution more balanced for model training.

In [14]:
print("skew of price      :", round(y_tr.skew(), 2))
print("skew of log(price) :", round(np.log1p(y_tr).skew(), 2))

skew of price      : 2.04
skew of log(price) : -0.4
